# 0. Antes de empezar

Definimos aquí algunas funciones y clases que utilizaremos en esta Notebook

In [ ]:
pip install rasterio

In [ ]:
# Funciones de visualización de la notebook anterior:
import numpy as np

def escalar_imagen(imagen, p, nodata = None):
  # Calcular los percentiles p y 100-p
  valor_min = np.percentile(imagen[imagen != nodata], p)
  valor_max = np.percentile(imagen[imagen != nodata], 100-p)

  # Truncar la imagen al rango [p1, p99]
  ecualizada = np.clip(imagen, valor_min, valor_max)

  # Normalizar la imagen al rango 0-255.
  ecualizada = (ecualizada - valor_min) / (valor_max - valor_min) * 255

  # Convertir la imagen ecualizada a entero de 8 bits
  ecualizada = np.uint8(ecualizada)

  # Retornar la imagen ecualizada
  return ecualizada

def escalar_imagen_multibanda(imagen, p, nodata = None):
  # Crear un array vacio con la misma estructura que imagen_spot_apilada
  imagen_escalada = np.empty_like(imagen)

  # Definir una variable con el número de canales
  canales = imagen_escalada.shape[-1]

  # Escalar una a una las bandas (usando p)
  for i in range(canales):
    canal = imagen[:, :, i]
    imagen_escalada[:, :, i] = escalar_imagen(canal, p, nodata = nodata)

  # Convertir a np.uint8 el array escalado (no hace falta seguir usando float)
  imagen_escalada = np.uint8(imagen_escalada)

  return imagen_escalada

# Notebook 02b: Vectores, Índices espectrales y escritura de archivos

### Datos con los que trabajaremos en esta notebook:

- Una imagen Sentinel-2 de la misma zona de estudio (**[S2_20180619_2-3-4-8-11-12.tif](https://drive.google.com/file/d/1dehVX2ClJBhEhGxrrVhvS5LH3SNK4RL0/view?usp=sharing)**) (misma que en la Notebook 02a).


Pueden descargarlas en sus equipos para visualizarlas localmente, o bien usar `gdown` para bajarlas directamente en la sesión de Colab.

## 1. Trabajando con datos vectoriales

A diferencia del modelo raster, en el modelo vectorial los datos no se organizan como una matriz, sino que adoptan la representación de elementos geométricos básicos como los puntos, las líneas o los polígonos. El elemento vectorial base es el punto; varios puntos unidos por segmentos definen una línea; y varias líneas y puntos se usan para definir polígonos.

Esta estructura impacta directamente en la forma en que estos elementos se pueden leer con Python. Hay muchas operaciones que vimos con las imágenes que no podemos aplicar sobre los vectores (como hacer _slicing_ de un aarreglo o el álgebra entre bandas). Sin embargo, la estructura y disposición de los datos vectoriales permiten hacer otros análisis que no son tan directos sobre los archivos raster. Un poco de todo esto repasaremos en las siguientes secciones.

Existen muchas librerías de Python que nos permiten trabajar con datos vectoriales: Nosotros trabajaremos con una llamada **Geopandas**.

Geopandas es una extension de la librería Pandas que nos permite trabajar con datos geoespaciales. Para ello toma elementos de otras librerías especialiazadas como **[shapely](https://shapely.readthedocs.io/en/stable/project.html)** para realizar operaciones geométricas, y **[fiona](https://fiona.readthedocs.io/en/latest/manual.html)** para acceder y/o modificar los archivos. Es decir, se trata de un paquete de funcionalidades de alto nivel.


### 1.1. Geopandas - Generación de un GeoDataFrame

Geopandas maneja dos tipos de estructuras de datos: _Geoseries_ y _Geodataframes_.  Estos son subclases de las Series y los Dataframes que maneja Pandas. Una _Geoseries_ es la estructura elemental en Geopandas. Se trata de un vector donde cada una de esas entradas comprende una o más formas o elementos geométricos. Cada uno de estos registros puede constar de una sola geometría (por ejemplo los polígonos de una capa de barrios de una ciudad) o muchas geometrías (por ejemplo, para representar un archipiélago se usan varios polígonos donde cada uno representa una isla):

![multiPol.png](https://drive.google.com/uc?id=12TXwOysaWmnOk6SlCV_WSrLGTPkV-E6u)

Las _Geoseries_ se pueden manipular como objetos de [Shapely](https://pypi.org/project/Shapely/), lo cual otorga un gran abanico de opciones. Todo el procesamiento geométrico que nos permite GeoPandas descansa sobre las capacidades que le otorga Shapely.

Los _Geodataframes_ tienen una estructura tabular y contienene una o más _Geoseries_, que representan cada una de las columnas de dicha tabla. Lo importante aquí es entender que dentro de un _Geodataframe_ siempre habrá una columna especial especial sobre la que se pueden hacer las operaciones geométricas. Esta columna normalmente es indicada como la geometría o _geometry_ , aunque su nombre identificatorio no sea exactamente ese:

![geoDataFrame.png](https://drive.google.com/uc?id=1jjAqK5cLQDE6M9olALWBe1h_5rVXF4rs)



Para comenzar, levantaremos nuestra imagen Sentinel-2 (archivo **S2_20180619_2-3-4-8-11-12.tif**) con Rasterio como en la Notebook 02a:

```python
import gdown
import rasterio

# ID del archivo de Google Drive
file_id = "1dehVX2ClJBhEhGxrrVhvS5LH3SNK4RL0"
url = f"https://drive.google.com/uc?id={file_id}"

# Descargar la imagen
output = "S2_20180619_2-3-4-8-11-12.tif"
gdown.download(url, output, quiet=False)

# Cargar la imagen
s2 = rasterio.open(output)
s2_bandas = s2.read()

# Escalar la imagen
s2_bandas_escalado = escalar_imagen_multibanda(s2_bandas.transpose(1,2,0), 1)
```

A continuación, vamos a generar un polígono a partir de coordenadas de las esquinas superior izquierda e inferior derecha, al que le vamos a establecer un sistema de referencia igual al de la imagen S2 sobre la que vamos a aplicar el recorte. Para ello, utilizaremos `geopandas` y `shapely`.

```python
from shapely.geometry import box
import geopandas as gpd

# Coordenadas de recorte
x_min, y_max = 394059, 6526889  # Superior izquierda
x_max, y_min = 397701, 6523349  # Inferior derecha

# Crear un polígono a partir de las coordenadas
poligono = gpd.GeoDataFrame(geometry=[box(x_min, y_min, x_max, y_max)], crs=s2.crs)
```
Una vez definido el polígono, vamos a imprimir algunas de sus propiedades como su extensión total, y su sistema de coordenadas:

``` python
print(poligono)
print(f'Tipo de objeto: {type(poligono)}')
print(f'Extensión del polígono: {poligono.total_bounds}')
print(f'Sistema de coordenadas del polígono: {poligono.crs}')

```

Si queremos ver donde se ubica el polígono definido sobre nuestra imagen, podemos graficar el raster con el polígono por encima de la siguiente forma. Para ello es necesario que tanto el polígono como el raster tengan el mismo sistema de referencia, lo que definimos en la celda anterior. Luego debemos usar el parámetro `extent` dentro de `imshow` para trabajar con coordenadas geográficas. A su vez, para plotear el polígono podemos mostrar solo sus bordes, usando `poligono.boundary.plot()`, de manera tal que no tape la imagen de fondo:

```python
import matplotlib.pyplot as plt

# Extent en coordenadas reales (xmin, xmax, ymin, ymax)
left, bottom, right, top = s2.bounds
extent = (left, right, bottom, top)

# Visualizar la imagen RGB
fig, ax = plt.subplots(1, 1, figsize=(10, 10))

# Mostrar la imagen en coordenadas reales
ax.imshow(s2_bandas_escalado[:, :, [2, 1, 0]], extent=extent, origin="upper")

# Dibujar polígono en coordenadas reales
poligono.boundary.plot(ax=ax, color="red", linewidth=2)

ax.set_title("Polígono sobre Imagen Sentinel-2")
plt.ticklabel_format(style="plain", axis="y") #Esto es para evitar la notación científica en las coordenadas. Por ejemplo, 6.5281e6
plt.show()
```

### 1.2. Recorte de la imagen usando un polígono rectangular

En la Clase 01 vimos una manera de realizar recortes de nuestras imágenes como _subsets_ rectangulares o cuadrados, aplicando el _slicing_ de Numpy. En esta clase veremos una manera de hacerlo más sencilla con Rasterio, y utilizando además polígonos de nuestros vectores. Este método conserva la georreferenciación de la imagen, que es necesaria cuando tenemos que guardarlo y usarlo en un software de Sistemas de Información Geográfica, por ejemplo.

Tenemos dos maneras de hacer nuestro "recorte":

A) Usando la extensión espacial del polígono (con lo cual obtendríamos un rectángulo o cuadrado)

B) Usando los límites exactos del polígono. En ese caso no sería solamente un recorte, sino una operación en **enmascarado**.

La extensión espacial se refiere siempre al borde o línea que une las coordenadas más extremas en Norte,Sur, Este y Oeste. La siguiente figura (extraída de [esta página](https://www.earthdatascience.org/courses/use-data-open-source-python/intro-raster-data-python/raster-data-processing/crop-raster-data-with-shapefile-in-python/)) lo muestra más claramente para el caso de vectores:

![spatialExtent.png](https://drive.google.com/uc?id=1Y6m9uFvx2agUWwMPBO_CPOy0iHL2uRMO)

Nuestro polígono es rectangular, con lo cual el recorte que haremos se corresponderá con _A_. Sin embargo, luego veremos cómo queda un recorte usando un polígono irregular de ejemplo, y como eso afecta a los valores sin dato. La función que usaremos para hacer el recorte es de `Rasterio` y se llama `mask`:

```python
from rasterio.mask import mask

s2_clip, clip_transform = mask(dataset=s2,shapes = poligono.geometry, crop = True)
print('Nuevo transform: ', clip_transform)
print('Dimensiones del recorte', s2_clip.shape)
print('Dimensiones del array original', s2_bandas.shape)
```


**Mask** nos devuelve el archivo recortado como arreglo de `NumPy` (lo hemos llamado **s2_clip**) y su _GeoTransform_ ya calculado (lo hemos asignado a la variable **clip_transform**) Las dimensiones de `s2_clip` son distintas a las que tiene la imagen original, porque usamos el parámetro `crop = True`. Por último, observar que en `dataset` pasamos como argumento la variable `s2` que es la imagen leída con `Rasterio`, antes de ser convertida a `array` de `NumPy`.

Si hacemos lo mismo pero con `crop = False`, visualmente puede parecer que se produce un recorte, pero en realidad lo que sucede es un **enmascarado** de la imagen:

```python
s2_mask, mask_transform = mask(dataset=s2,shapes = poligono.geometry, crop = False)
print('Nuevo transform: ', mask_transform)
print('Dimensiones del enmascarado', s2_mask.shape)
print('Dimensiones del array original', s2_bandas.shape)
```


Este `array` enmascarado es igual en tamaño al original, porque se usó `crop = False`. A todos los píxeles que caen afuera del rectángulo se les asignó el valor `0`, que se espeficia en el parámetro `nodata`. Veamos esto en un ploteo para entender las diferencias entre ambos:

```python
# Escalar las imágenes para que se vean mejor (atención, que en el cas ode s2_mask debemos especificaar el nodata)
s2_clip_escalado = escalar_imagen_multibanda(s2_clip.transpose(1,2,0), 1)
s2_mask_escalado = escalar_imagen_multibanda(s2_mask.transpose(1,2,0), 1, nodata = 0)

from matplotlib import pyplot as plt
fig, ax = plt.subplots(1,2,figsize = (12,6), gridspec_kw={'width_ratios': [2, 1]})
ax[0].imshow(s2_mask_escalado[:,:,[2,1,0]])
ax[0].set_title("Enmascarado")
ax[0].axis("off")
ax[1].imshow(s2_clip_escalado[:,:,[2,1,0]])
ax[1].set_title("Recortado")
ax[1].axis("off")
plt.show()
```

Efectivamente, en cada caso tenemos `arrays` de distinto tamaño, y también con un origen geográfico distinto. Mientras que `s2_mask` sigue ocupando la misma extensión que la imagen original, `s2_mask` tiene unas coordenadas de origen distintas. Esto hay que tenerlo en cuenta si vamos a guardar un recorte (sección 3), y lo podemos verificar imprimiendo las variables `clip_transform` y `mask_transform` generadas por `mask`:

```python
print("Transform del clip:\n")
print(clip_transform)
print("Transform del enmascarado:\n")
print(mask_transform)
```

### 1.3. Recorte de la imagen usando un polígono irregular

Muchas veces, cuando trabajamos con Sistemas de Información Geográfica, es habitual que contemos con algún polígono ya digitalizado y que lo necesitemos para enmascarar y recortar nuestra imagen. Vamos a trabajar con el siguiente polígono de ejemplo, resaltado en amarillo en la captura, y reproducir los pasos anteriores, para entender cómo quedaría nuestra imagen de salida.

![radioMask.png](https://drive.google.com/uc?id=1a48YBNn7DNAwNC1zhueFf8IvsklpF5iV)

```python
# ID del archivo de Google Drive
file_id = "1bBKduhO8ZnOS8Hkm6QDjIEXjbLfdDcjS"
url = f"https://drive.google.com/uc?id={file_id}"

# Descargar la imagen
output = "poligono_irregular.gpkg"
gdown.download(url, output, quiet=False)

# Importar la librería geopandas
import geopandas as gpd

poligono_irregular = gpd.read_file(output)
poligono_irregular.plot()
```


A continuación, aplicaremos `mask` con `crop = True` para enmascarar y recortar la imagen. Especificaremos `nodata = 0` para los píxeles que quedan afuera de los límites del polígono.

```python
# Aplicamos el enmascarado con el polígono irregular
s2_mask_irregular, mask_transform_irregular = mask(dataset=s2,shapes = poligono_irregular.geometry, crop = True, nodata = 0)

#Aplicamos el escalado para la visualización
s2_mask_irregular_escalado = escalar_imagen_multibanda(s2_mask_irregular.transpose(1,2,0), 1, nodata = 0)

# Visualizamos el resultado
plt.figure(figsize = (12,6))
plt.imshow(s2_mask_irregular_escalado[:,:,[2,1,0]])
plt.axis("off")
plt.show()

print("Transform de la nueva imagen:\n")
print(mask_transform_irregular)
```



El resultado de este enmascarado es similar a los anteriores, pero no nos ha quedado una imagen con _forma rectangular_, sino que es una matriz donde los datos de relleno corresponden a las zonas que quedaron afuera del polígono.

## 2. Cálculo de índices espectrales

Seguiremos trabajando con `s2_clip` (la imagen extraída a partir del rectángulo), y la utilizaremos para calcular índices espetrales con sus bandas, aplicando lo visto en el teórico. Al ser arreglos de Numpy, se pueden realizar operaciones matemáticas entre las bandas de la imagen (sumarlas, multiplacarlas, restarlas, dividirlas, normalizarlas por una constante, etc).

Aprovecharemos estas capacidades para calcular dos índices normalizados que son de gran uso en la Teledetección Óptica: El Índice de Vegetación Normalizado (o _NDVI_ por sus siglas en inglés), y el Índice de Agua Normalizado (o _NDWI_ por sus siglas en inglés).

### 2.1. NDVI

Empecemos con el _NDVI_:


$ NDVI = \frac{NIR - R}{NIR + R} $  

Donde $NIR$ se refiere a la banda del Infrarrojo Cercano, y $R$ se refiere a la banda del Rojo (¿Cuáles serían en nuestra imagen Sentinel?).

```python
# Banda del NIR
b_nir = s2_clip[3].astype(float)

# Banda del R
b_red = s2_clip[2].astype(float)

ndvi = (b_nir - b_red) / (b_nir + b_red)
```
Observar que hemos agregado `.astype(float)`. Esto lo debemos hacer cuando trabajamos con datos enteros y hacemos operaciones que pueden dar como resultado números flotantes. Si no lo hacemos, tendremos valores de salida erróneos.

Luego, podemos plotear el nuevo índice junto a la combinación RGB que resalta la vegetación, para corroborar que en las zonas donde los píxeles se ven más rojos, el NDVI es más alto:

```python
fig, ax = plt.subplots(1, 3, figsize=(18, 6), gridspec_kw={'width_ratios': [1.1, 1, 0.8]})

# Primer subplot: NDVI en escala de grises
ax[0].imshow(ndvi, cmap='gray')
ax[0].set_title("NDVI")
ax[0].axis("off")
fig.colorbar(ax[0].imshow(ndvi, cmap='gray'), ax=ax[0], fraction=0.046, pad=0.04)

# Segundo subplot: Imagen RGB del clip escalado
ax[1].imshow(s2_clip_escalado[:, :, [3, 2, 1]])
ax[1].set_title("Combinación falso color (vegetación)")
ax[1].axis("off")

# Tercer subplot: Histograma del NDVI
ax[2].hist(ndvi.ravel(), bins=250, alpha=0.7)
ax[2].set_title("Histograma NDVI")
ax[2].set_xlabel("Valor NDVI")
ax[2].set_ylabel("Frecuencia")

plt.tight_layout()
plt.show()
```

### 2.2. Calcular el NDWI

Replicaremos los pasos realizados con el NDVI, para calcular el Ìndice Normalizado de Agua (o NDWI) según la siguiente fórmula:

$ NDWI = \frac{G - NIR}{G + NIR} $

Donde $ G $ corresponde a la banda del verde, y $NIR$ es la misma que utilizamos para el NDVI

**Nota**: La fórmula propuesta para el NDWI corresponde a la formulación de [Mc. Feeters (1996)](https://hess.copernicus.org/articles/22/4349/2018/hess-22-4349-2018-t01.pdf)

En el código de abajo, completá la información de las bandas que tenés que usar para calcular este índice, de manera similar a como está hecho con el NDVI.

```python
# Banda del NIR
b_nir = # completar

# Banda del G
b_green = # completar

ndwi = (b_green - b_nir) / (b_green + b_nir)
```

Y a continuación podemos visualizar el índice junto a alguna combinación RGB que nos ayude a interpretar:

```python
fig, ax = plt.subplots(1, 3, figsize=(18, 6), gridspec_kw={'width_ratios': [1.1, 1, 0.8]})

# Primer subplot: NDWI en escala de grises
ax[0].imshow(ndvi, cmap='gray')
ax[0].set_title("NDWI")
ax[0].axis("off")
fig.colorbar(ax[0].imshow(ndwi, cmap='gray'), ax=ax[0], fraction=0.046, pad=0.04)

# Segundo subplot: Imagen RGB del clip escalado
ax[1].imshow(s2_clip_escalado[:, :, [4, 3, 2]])
ax[1].set_title("Combinación falso color (vegetación)")
ax[1].axis("off")

# Tercer subplot: Histograma del NDWI
ax[2].hist(ndwi.ravel(), bins=250, alpha=0.7)
ax[2].set_title("Histograma NDWI")
ax[2].set_xlabel("Valor NDWI")
ax[2].set_ylabel("Frecuencia")

plt.show()
```

### 2.3. Binarizar el NDWI y generar una máscara de agua-no agua

Vamos a aprovechar el mapa de NDWI para intentar binarizarlo y extraer una máscara donde 1 (`True`) representa Agua y 0 (`False`) representa el resto. Mirando el histograma, un valor razonable podría ser 0.2:

```python
# Construimos la máscara con umbral 0.2
mascara_agua = ndwi > 0.2

# La pasamos a valores 0-1
mascara_agua = mascara_agua*1

fig, ax = plt.subplots(1, 3, figsize=(18, 6), gridspec_kw={'width_ratios': [1.1, 1, 1]})

# Primer subplot: NDWI en escala de grises
im0 = ax[0].imshow(ndwi, cmap='gray')
ax[0].set_title("NDWI")
ax[0].axis("off")
fig.colorbar(im0, ax=ax[0], fraction=0.046, pad=0.04)

# Segundo subplot: Imagen RGB del clip escalado
ax[1].imshow(s2_clip_escalado[:, :, [4, 3, 2]])
ax[1].set_title("Combinación falso color (vegetación)")
ax[1].axis("off")

# Tercer subplot: Máscara de Agua
ax[2].imshow(mascara_agua)
ax[2].set_title("Máscara agua-no agua")
ax[2].axis("off")

plt.tight_layout()
plt.show()
```

## 3. Escritura de los archivos

Para finalizar esta Notebook, guardaremos como archivo vectorial el rectángulo generado para extraer el recorte, y como archivo GeoTiff la imagen recortada.

### 3.1. Guardar el archivo vectorial

El guardado de archivos vectoriales con `Geopandas` es muy simple. Hay que utilizar el comando [`to_file()`](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.to_file.html) y especificar el nombre y ruta del archivo de salida. El formato de archivo de salida puede ser cualquiera de los _drivers_ vectroriales especificados en la [página de GDAL](https://gdal.org/en/stable/drivers/vector/index.html); nosotros utilizaremos el formato _Geopackage_ (extensión .gpkg).

```python
# Guardar el archivo
poligono.to_file('rectangulo_clase02.gpkg')
```

El archivo lo verán disponible en la sección de archivos de Colab. Descargarlo antes de cerrar esta sesión, para que quede en el equipo local de cada uno.

![radioMask.png](https://drive.google.com/uc?id=1tPpbVOBBH_TQimg7-r1UyQLW0mfeedl3)

### 3.2. Guardar el archivo ráster o imagen

Para guardar imágenes con `rasterio` lo más importante es tener bien definidos los metadatos de salida. Nosotros tenemos definidos por ejemplo los metadatos de la imagen original:

```python
print(s2.meta)
```

Si por ejemplo nosotros quisiéramos guardar `s2_clip`, la imagen recortada con el rectángulo, en los metadatos tendríamos que redefinir sus dimensiones (`width` y `height`), y su geotrasnformación, porque su esquina superior izquierda es distinta a la imagen original. Nosotros hemos capturado eso en la variable `clip_transform`.Por otro lado, las dimensiones las podemos sacar de `shape`:

```python
print("Transform de s2_clip:\n", clip_transform)
print("Ancho (Width) de s2_clip:\n", s2_clip.shape[2])
print("Alto (Height) de s2_clip:\n", s2_clip.shape[ 1])
```

Usando estas variabales podemos definir unos nuevos metadatos para guardar `s2_clip` como archivo GeoTiff afuera de Colab:+

```python
from rasterio.crs import CRS

clip_metadatos = {'driver': 'GTiff',
                  'dtype': 'uint16',
                  'nodata': None,
                  'width': s2_clip.shape[2],
                  'height': s2_clip.shape[1],
                  'count': 6,
                  'crs': CRS.from_wkt('PROJCS["WGS 84 / UTM zone 20S",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",-63],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",10000000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","32720"]]'),
                  'transform': clip_transform}
```

Con estos nuevos metadatos definidos, usamos el siguiente comando para generar nuestro archivo y pasarle la información de `s2_clip` y `clip_metadatos`:

```python
with rasterio.open('s2_clip.tif', 'w', **clip_metadatos) as dst:
    dst.write(s2_clip)
```

El uso de `with` en el bloque anterior permite un manejo más eficiente de los recursos del sistema. Una vez que se ejecuta la operación que está dentro del contexto definido, se liberan todos los recursos y se termina de escribir el archivo. Si lo hiciéramos sin `with`, la manera sería:

```python
dst = rasterio.open('s2_clip.tif', `w`, **clip_metadata)
dst.write(s2_clip)
del dst
```

Una vez que esté generado el archivo, descargarlo al igual que con el caso del vector:

![radioMask.png](https://drive.google.com/uc?id=19VJRaIdt5_J3zHXLJ9D3CYyGnMe0B5_T)

Si quisiéramos guardar por ejemplo el NDVI, podríamos usar los mismos metadatos, pero tendríamos que cambiar el tipo de dato, ya que `uint16` no soporta valores flotantes o decimales. También deberíamos modificar el parámetro `count`, ya que no son más 6 bandas, sino 1.

Además, la función `write()` esoera como entrada un `array` de 3 dimensiones `(bandas, filas, columnas)`, pero `ndvi` tiene solo 2, `(filas,columnas)`. Lo que se puede hacer es pasarle a `write` el mismo `ndvi` pero _expandido_ en 1 dimensión, usando `np.expand_dims(ndvi, axis=0)`:

```python
clip_metadatos_ndvi = {'driver': 'GTiff',
                  'dtype': 'float32',
                  'nodata': None,
                  'width': s2_clip.shape[2],
                  'height': s2_clip.shape[1],
                  'count': 1,
                  'crs': CRS.from_wkt('PROJCS["WGS 84 / UTM zone 20S",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",-63],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",10000000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","32720"]]'),
                  'transform': clip_transform}

with rasterio.open('ndvi_clip.tif', 'w', **clip_metadatos_ndvi) as dst:
    dst.write(np.expand_dims(ndvi, axis=0))
```



Una vez que esté generado el archivo, descargarlo al igual que con el caso del vector y de `s2_clip`:

![radioMask.png](https://drive.google.com/uc?id=1wg-MCAbz0MCX8iYaEzq71y-J3qxRascQ)

Guardá estos archivos en tu computadora, porque los usaremos para la Clase 03.

### 3.4. ¿Qué pasa con los valores 'nodato'?

Para cerrar, vamos a guardar como GeoTiff a `s2_mask_irregular`, la imagen que generamos usando como máscara un vector externo. Cuando tenemos píxeles que quedan enmascarados, los cuales rellenamos con 0 en este caso, si queremos que un software de procesmaiento de imágenes los reconozca como tal y los muestre _transparentes_, debemos especificar este valor 0 como `nodata` en los metadatos.

```python
mask_metadatos_irregular = {'driver': 'GTiff',
                  'dtype': 'float32',
                  'nodata': None,
                  'width': s2_mask_irregular.shape[2],
                  'height': s2_mask_irregular.shape[1],
                  'count': 6,
                  'crs': CRS.from_wkt('PROJCS["WGS 84 / UTM zone 20S",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",-63],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",10000000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","32720"]]'),
                  'transform': mask_transform_irregular}

with rasterio.open('s2_mask_irregular.tif', 'w', **mask_metadatos_irregular) as dst:
    dst.write(s2_mask_irregular)
```


Si nosotros no especificamos el el valor `nodata` es 0, en otros programas como QGIS la imagen se verá con sus bordes negros como en la imagen de la derecha.

![radioMask.png](https://drive.google.com/uc?id=1tchpSjlvCGPPRv6CVIDdLqV_yAqOzjFX)

### 3.4. Ejercicio

Siguiendo los pasos anteriores, guardá como GeoTiff la variable `ndwi`, replicando la misma lógica que usamos con el NDVI. También, probá guardar como archivo la máscara binaria generada a partir de ese índice. ¿Qué tipo de dato te conviene usar si son todos valores `0`y `1`?